In [ ]:
from pathlib import Path
import os, sys
_candidates = [Path(os.environ["MIRISA_ROOT"])] if "MIRISA_ROOT" in os.environ else [Path.cwd(), *Path.cwd().parents]
_candidates += [p / "INDEX_MIRISA" for p in list(_candidates)]
_root = next((p.resolve() for p in _candidates if (p / "config/paths.R").is_file()), None)
if _root is None:
    raise RuntimeError("Não encontrei INDEX_MIRISA; define MIRISA_ROOT.")
sys.path.insert(0, str(_root / "scripts/functions/PYTHON"))
from project_paths import setup_project
setup_project(_root)


Avaliação de Aspectos Climatologicos do MIRI.SA 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm

import datetime
import time as time
import os.path
import pandas as pd
from datetime import date
import matplotlib.patches as mpatches


In [ ]:
output_figure = "outputs/figures/MIRISA_amplitude_climatology_19910401_20210331.png"

input_csv = "outputs/tables/MIRISA_CLIM_19910401_20210331_CREATE_20260314.txt"
start_date = "1991-04-01"
end_date   = "2021-03-31"

# =========================
# LEITURA DOS DADOS
# =========================
# Esperado:
# "Year";"Month";"Day";"MIRISA1c1";"MIRISA1c2";"Phase";"Amplitude"

df = pd.read_csv(
    input_csv,
    sep=";",
    quotechar='"',
    engine="python"
)

# Remove espacos e aspas residuais nos nomes das colunas
df.columns = [c.strip().replace('"', "") for c in df.columns]

# Garante tipos numericos
numeric_cols = ["Year", "Month", "Day", "MIRISA1c1", "MIRISA1c2", "Phase", "Amplitude"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Monta a coluna de data
df["date"] = pd.to_datetime(
    dict(year=df["Year"], month=df["Month"], day=df["Day"]),
    errors="coerce"
)
# Remove linhas invalidas

df = df.dropna(subset=["date", "Amplitude", "Phase"]).copy()

mask = (df["date"] >= start_date) & (df["date"] <= end_date)
df = df.loc[mask].copy()
# =========================
# FILTRO DO PERIODO
# =========================
mask = (df["date"] >= pd.Timestamp(start_date)) & (df["date"] <= pd.Timestamp(end_date))
clim = df.loc[mask].copy()

if clim.empty:
    raise ValueError("Nenhum dado encontrado no periodo selecionado.")

# =========================
# ESTAÇÕES
# =========================
def season(m):
    if m in [12,1,2]: return "DJF"
    if m in [3,4,5]: return "MAM"
    if m in [6,7,8]: return "JJA"
    return "SON"

clim["season"] = clim["Month"].apply(season)
season_order = ["DJF","MAM","JJA","SON"]

# =========================
# CATEGORIAS DE AMPLITUDE
# =========================
def classify_amplitude(x):
    if x < 0.8:
        return "Inactive (<0.8)"
    elif x < 1.2:
        return "Weakly Active (0.8-1.2)"    
    elif x < 1.6:
        return "Moderately Active (1.2-1.6)"
    elif x < 2.0:
        return "Very Active (1.6-2.0)"
    else:
        return "Extremely Active (≥2.0)"

clim["amp_class"] = clim["Amplitude"].apply(classify_amplitude)

class_order = [
    "Inactive (<0.8)",
    "Weakly Active (0.8-1.2)",
    "Moderately Active (1.2-1.6)",
    "Very Active (1.6-2.0)",
    "Extremely Active (≥2.0)"
]

class_counts = clim["amp_class"].value_counts().reindex(class_order, fill_value=0)
class_percent = 100.0 * class_counts / class_counts.sum()

class_order1 = [
    "inactive \n (<0.8)",
    "weakly active \n (0.8-1.2)",
    "moderately active \n (1.2-1.6)",
    "very active \n (1.6-2.0)",
    "extremely active \n (≥2.0)"
]

# =========================
# PAINEL (a): sazonal
# =========================
seasonal = (
    clim.groupby(["season","amp_class"])
      .size()
      .unstack(fill_value=0)
      .reindex(index=season_order, columns=class_order)
)

seasonal_pct = seasonal.div(seasonal.sum(axis=1), axis=0)*100


# =========================
# DENSIDADE (sem scipy)
# =========================
# Histograma normalizado com suavizacao simples via media movel.
# Funciona bem o bastante sem depender de scipy, porque a vida já complica sozinha.
hist_counts, bin_edges = np.histogram(clim["Amplitude"], bins=40, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
cores=["royalblue", "green", "orange","red", "darkred"]
# suavizacao simples
window = 5
kernel = np.ones(window) / window
smooth_density = np.convolve(hist_counts, kernel, mode="same")

# =========================
# FIGURA
# =========================
#fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig = plt.figure(figsize=(16,5))
gs = fig.add_gridspec(1, 3, width_ratios=[1,1,2])

axes0 = fig.add_subplot(gs[0,0])
axes1 = fig.add_subplot(gs[0,1])
axes2 = fig.add_subplot(gs[0,2])

# Painel 1: histograma
axes0.hist(clim["Amplitude"], bins=40, density=True, color=cores[0]) #edgecolor="black")
axes0.plot(bin_centers, smooth_density, linewidth=2,color="darkred")
axes0.axvline(0.8, linestyle="--", linewidth=1,color="red")
axes0.axvline(1.2, linestyle="--", linewidth=1,color="red")
axes0.axvline(1.6, linestyle="--", linewidth=1,color="red")
axes0.axvline(2.0, linestyle="--", linewidth=1,color="red")
axes0.set_title("a) Histogram of MIRISA amplitude")
axes0.set_xlabel("Amplitude")
axes0.set_ylabel("Frequency")

# Painel 2: porcentagem por classe
axes1.barh(class_order, class_percent.values, edgecolor="black",facecolor=cores[0])
axes1.set_title("b) Percentage of days by amplitude class")
axes1.set_xlim([0,65])
axes1.set_xlabel("Percentage (%)")
axes1.set_ylabel("Amplitude class")
axes1.tick_params(axis="x")
axes1.set_yticklabels(class_order1)

for i, v in enumerate(class_percent.values):
    axes1.text(v + 5,i, f"{v:.1f}%", ha="center", va="bottom", fontsize=10)

# Titulo geral
fig.suptitle(
    "MIRISA amplitude climatology (1991-04-01 to 2021-03-31)",
    fontsize=14,
    y=1.02
)
width = 0.18
x = np.arange(len(season_order))

for i, cls in enumerate(class_order):
    axes2.bar(
        x + i*width,
        seasonal_pct[cls].values,
        width=width,facecolor=cores[i],edgecolor="k",
        label=cls
    )

axes2.set_xticks(x + width*1.5)
axes2.set_xticklabels(season_order)
axes2.set_ylim([0,60])

axes2.set_title("c) Seasonal variability of MIRISA amplitude")
axes2.set_ylabel("Percentage (%)")
axes2.set_xlabel("Season")
axes2.legend(frameon=False,loc='upper left',ncol=2)

fig.tight_layout()
fig.savefig(output_figure, dpi=300, bbox_inches="tight")



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
input_file ="outputs/tables/MIRISA_CLIM_19910401_20210331_CREATE_20260314.txt"
output_figure = "outputs/figures/MIRISA_seasonal_phase_amplitude.png"

start_date = "1991-04-01"
end_date   = "2021-03-31"

# =========================
# LEITURA
# =========================
df = pd.read_csv(input_file, sep=";", quotechar='"', engine="python")
df.columns = [c.strip().replace('"', "") for c in df.columns]

num_cols = ["Year", "Month", "Day", "Phase", "Amplitude"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["date"] = pd.to_datetime(
    dict(year=df["Year"], month=df["Month"], day=df["Day"]),
    errors="coerce"
)

df = df.dropna(subset=["date", "Amplitude", "Phase"]).copy()

mask = (df["date"] >= start_date) & (df["date"] <= end_date)
df = df.loc[mask].copy()

# =========================
# CATEGORIAS DE AMPLITUDE
# =========================
def classify_amp(x):
    if x < 0.8:
        return "Inactive"
    elif x < 1.2:
        return "Weakly Active"
    elif x < 1.6:
        return "Moderately Active"
    elif x < 2.0:
        return "Very Active"
    else:
        return "Extremely Active"

df["amp_class"] = df["Amplitude"].apply(classify_amp)

class_order = ["Inactive", "Weakly Active", "Moderately Active", "Very Active","Extremely Active"]

# =========================
# ESTAÇÕES
# =========================
def season(m):
    if m in [12,1,2]: return "DJF"
    if m in [3,4,5]: return "MAM"
    if m in [6,7,8]: return "JJA"
    return "SON"

df["season"] = df["Month"].apply(season)
season_order = ["DJF","MAM","JJA","SON"]

# =========================
# PAINEL (a): sazonal
# =========================
seasonal = (
    df.groupby(["season","amp_class"])
      .size()
      .unstack(fill_value=0)
      .reindex(index=season_order, columns=class_order)
)

seasonal_pct = seasonal.div(seasonal.sum(axis=1), axis=0)*100

# =========================
# PAINEL (b): fase x amplitude
# =========================
phase_amp = (
    df.groupby(["Phase","amp_class"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=class_order)
)

phase_amp_pct = phase_amp.div(phase_amp.sum(axis=1), axis=0)*100

# =========================
# FIGURA
# =========================
# =========================
# FIGURA
# =========================
fig = plt.figure(figsize=(10,5))
 
# largura das barras
width = 0.18

# -------------------------
# Painel (a) - sazonal
# -------------------------
cores=["royalblue", "green", "orange","red", "darkred"]
     
# -------------------------
# Painel (b) - fase x amplitude
# -------------------------
phases = phase_amp_pct.index.values
x2 = np.arange(len(phases))

for i, cls in enumerate(class_order):
    plt.bar(
        x2 + i*width,
        phase_amp_pct[cls].values,
        width=width,
        label=cls,facecolor=cores[i],edgecolor="k"
    )

plt.xticks(x2 + width*1.5,phases)
#plt.xticklabels(phases)
plt.ylim([0,60])
plt.title("Phase–amplitude relationship")
plt.xlabel("Phase")
plt.ylabel("Percentage (%)")

# remove legenda duplicada
plt.legend(frameon=False,ncol=5)

# -------------------------
# FINAL
# -------------------------
fig.suptitle("MIRISA seasonal variability and phase–amplitude coupling", y=1.02)
fig.tight_layout()
fig.savefig(output_figure, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================
# CONFIG
# =========================================
input_file = "outputs/tables/MIRISA_CLIM_19910401_20210331_CREATE_20260314.txt"   # troque pelo nome real
output_figure = "outputs/figures/MIRISA_phase_amplitude_seasonal_polar.png"

start_date = "1991-04-01"
end_date   = "2021-03-31"

# Classes de amplitude
amp_classes = [
    ("Inactive", 0.0, 0.8, "royalblue"),
    ("Weakly Active", 0.8, 1.2, "green"),
    ("Moderately Active", 1.2, 1.6, "orange"),
    ("Very Active", 1.6, 2.0, "red"),
    ("Extremely Active", 2.0, np.inf, "darkred"),
]

season_order = ["JJA", "SON", "DJF", "MAM"]

# =========================================
# LEITURA
# =========================================
df = pd.read_csv(input_file, sep=";", quotechar='"', engine="python")
df.columns = [c.strip().replace('"', "") for c in df.columns]

num_cols = ["Year", "Month", "Day", "Phase", "Amplitude"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["date"] = pd.to_datetime(
    dict(year=df["Year"], month=df["Month"], day=df["Day"]),
    errors="coerce"
)

df = df.dropna(subset=["date", "Phase", "Amplitude"]).copy()

mask = (df["date"] >= pd.Timestamp(start_date)) & (df["date"] <= pd.Timestamp(end_date))
df = df.loc[mask].copy()

# Mantém só fases 1-8
df = df[df["Phase"].isin(range(1, 9))].copy()

# =========================================
# ESTAÇÕES
# =========================================
def get_season(month):
    if month in [12, 1, 2]:
        return "DJF"
    elif month in [3, 4, 5]:
        return "MAM"
    elif month in [6, 7, 8]:
        return "JJA"
    else:
        return "SON"

df["Season"] = df["Month"].apply(get_season)

# =========================================
# CLASSIFICAÇÃO DA AMPLITUDE
# =========================================
def classify_amplitude(x):
    for label, lo, hi, color in amp_classes:
        if lo <= x < hi:
            return label
    return np.nan

df["amp_class"] = df["Amplitude"].apply(classify_amplitude)

class_order = [c[0] for c in amp_classes]
class_colors = {c[0]: c[3] for c in amp_classes}

# =========================================
# TABELA DE FREQUÊNCIA RELATIVA (%)
# Percentual em cada fase por estação
# =========================================
# Conta quantos dias há em cada combinação season-phase-class
counts = (
    df.groupby(["Season", "Phase", "amp_class"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=class_order, fill_value=0)
)

# Total por estação, para converter em frequência relativa dentro da estação
season_totals = df.groupby("Season").size()

# Cria estrutura para armazenar percentuais
percent = {}
for season in season_order:
    season_df = counts.loc[season] if season in counts.index.get_level_values(0) else None
    if season_df is None:
        percent[season] = pd.DataFrame(
            0, index=range(1, 9), columns=class_order, dtype=float
        )
        continue

    season_df = season_df.reindex(range(1, 9), fill_value=0)
    season_pct = season_df / season_totals.loc[season] * 100.0
    percent[season] = season_pct


In [ ]:
print(percent["DJF"])

In [ ]:

# =========================================
# FUNÇÃO PARA PAINEL POLAR
# =========================================


def plot_polar_panel(ax, season, panel_label,fs):
    data = percent[season]

    nphases = 8
    theta = np.linspace(0, 2 * np.pi, nphases, endpoint=False)
    theta = theta + (np.pi / nphases) + np.pi/4

    # largura angular de cada barra
    width = 2 * np.pi / nphases * 0.45

    # coloca fase 1 no quadrante inferior esquerdo, aproximando o estilo do paper
    ax.set_theta_offset((5 * np.pi / 4)-np.pi/2)
    ax.set_theta_direction(1)

    # barras empilhadas radialmente
    bottom = np.zeros(nphases)
    for cls in class_order:
        vals = data[cls].values
        ax.bar(
            theta,
            vals,
            width=width,
            bottom=bottom,
            color=class_colors[cls],
            edgecolor="none",
            align="center",
            label=cls
        )
        bottom += vals

    # grades radiais
    ax.set_ylim(0, 20)
    ax.set_yticks([5, 10, 15])
    ax.set_yticklabels(["5%", "10%", "15%"], fontsize=fs)
    ax.set_rlabel_position(180)

    # remove labels automáticos do theta
    ax.set_xticks([])

    # linhas divisórias das 8 fases
    for ang in np.linspace(0, 2*np.pi, 4, endpoint=False):
        ax.plot([ang, ang], [0, 20], color="0.65", lw=1,linestyle="dotted")
        ax.plot([(ang+np.pi/4), (ang+np.pi/4)], [0, 20], color="0.25", lw=1.1,linestyle="-")
        # ax.set_theta_offset((5 * np.pi / 4))
        # ax.set_theta_direction(1)

    # # números das fases
    for i, ang in enumerate(theta, start=1):
        ax.text(
            ang,
            22,
            str(i), # if i != 0 else str(8),
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold"
        )


    sector_labels = {
    "MJO: Western Pacific\nRain: Western Amazon": (theta[5]+theta[6])/2,
    "MJO: Maritime Continent\nRain: Wertern Brazil - South Brazil": (theta[1]+theta[2])/2,
    "MJO: Indian Ocean\nRain: Center S. America": (np.pi/4) , #((theta[7]-theta[0])/3)
    "MJO: West. Hem. & Africa\nRain: Northeast Brazil - South Brazil": (theta[3]+theta[4])/2
    }
    
    for txt, ang in sector_labels.items():

        ang_deg = np.degrees(ang) % 360

    # define rotação por quadrante (sentido horário)
        if 45 <= ang_deg < 135:
            rot = 90
        elif 135 <= ang_deg < 225:
            rot = 0 # Maritime continente
        elif 225 <= ang_deg <314:
            rot = 270 # Africa       
        elif 314 <= ang_deg <=315:
            rot = 0 # Africa       
        else:
            rot = 90  # Western Pacifico


        ax.text(
            ang,
            24.0,
            txt,
            ha="center",
            va="center",
            fontsize=fs,
            rotation=rot,
            rotation_mode="anchor" #,
            #bbox=dict(boxstyle="square,pad=0.15", fc="white", ec="0.7", lw=0.8)
        )

    # título da estação
    ax.text(
        0.96, 0.94, season,
        transform=ax.transAxes,
        ha="right", va="center",
        fontsize=1.1*fs
    )

    # # rótulo do painel
    ax.text(
        -0.2, 1.15, panel_label,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=1.1*fs,
        fontweight="bold"
    )

# =========================================
# FIGURA
# =========================================
fig, axes = plt.subplots(
    2, 2,
    figsize=(20, 20),
    subplot_kw=dict(projection="polar")
)

fig.subplots_adjust(
    left=0.10,
    right=0.9,
    top=0.90,
    bottom=0.15,
    wspace=0.2,
    hspace=0.40
)

panel_labels = ["a) JJA", "b) SON", "c) DJF", "d) MAM"]
fs = 19
for ax, season, plab in zip(axes.ravel(), season_order, panel_labels):
    plot_polar_panel(ax, season, plab,fs)

# legenda central
handles = [
    plt.Rectangle((0, 0), 1.2, 1.2, color=class_colors[c])
    for c in reversed(class_order)
]
labels = [
    "≥2.0",
    "1.6–2.0",
    "1.2–1.6",
    "0.8–1.2",
    "0–0.8"
]

fig.legend(
    handles,
    labels,
    title="Amplitude",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.07),
    ncol=5,   # importante!
    frameon=False,
    fontsize=20,
    title_fontsize=20
)

#fig.suptitle("MIRISA amplitude", fontsize=25, y=0.95)
# plt.tight_layout(rect=[0.15, 0.15, 0.95, 0.95])
plt.savefig(output_figure, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figura salva em: {output_figure}")

In [ ]:
nphases = 4
theta = np.linspace(0, 2 * np.pi, nphases, endpoint=False)
theta = theta + (np.pi / nphases)
print(theta)
xx1 = theta + np.pi/4
print(np.degrees(xx1))
print(np.degrees(theta))